# DeepSeis — Fallback Demo Notebook

Runs the full story end-to-end from already-trained checkpoints in `runs/default/`
(produced by `python -m deepseis.train --config configs/default.yaml`), for use if
the live Streamlit dashboard breaks on stage during the pitch.

1. Load noisy input + trained models
2. Denoise (fault-preservation ON vs OFF)
3. The "money shot": toggle comparison
4. Fault segmentation delta (noisy vs denoised)
5. F-K spectrum + local-similarity diagnostics

In [ ]:
import sys
sys.path.insert(0, "..")
import numpy as np
import torch
import matplotlib.pyplot as plt
from pathlib import Path

from deepseis.train import load_config, get_device, run_denoiser_inference, run_faultseg_inference
from deepseis.models.unet import DenoiserUNet
from deepseis.models.faultseg import FaultSegNet2D
from deepseis import metrics as metrics_mod
from deepseis.losses.frequency import fk_spectrum
from deepseis.interpretation.horizon import track_horizons

cfg = load_config("../configs/default.yaml")
run_dir = Path("..") / cfg["output"]["run_dir"]
device = get_device(cfg)
print("device:", device, " run_dir:", run_dir)

## 1. Load noisy input + trained models

In [ ]:
clean = np.load(run_dir / "clean.npy")
noisy = np.load(run_dir / "noisy.npy")
fault_mask = np.load(run_dir / "fault_mask.npy")

model_on = DenoiserUNet(**cfg["model"]["denoiser"]).to(device)
model_on.load_state_dict(torch.load(run_dir / cfg["output"]["checkpoint_name"], map_location=device))
model_on.eval()

model_off = DenoiserUNet(**cfg["model"]["denoiser"]).to(device)
model_off.load_state_dict(torch.load(run_dir / ("off_" + cfg["output"]["checkpoint_name"]), map_location=device))
model_off.eval()

faultseg = FaultSegNet2D(in_channels=1, base_channels=cfg["faultseg"]["base_channels"],
                          depth=cfg["faultseg"]["depth"]).to(device)
faultseg.load_state_dict(torch.load(run_dir / cfg["output"]["faultseg_checkpoint_name"], map_location=device))
faultseg.eval()
print("loaded: clean/noisy sections + denoiser (ON/OFF) + FaultSeg head")

## 2. Denoise

*"You can never record a seismic shot without noise — so there's no clean data to train on.
Here's how we denoise anyway."*

In [ ]:
denoised_on = run_denoiser_inference(model_on, noisy, cfg, device)
denoised_off = run_denoiser_inference(model_off, noisy, cfg, device)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, img, title in zip(axes, [noisy, denoised_off, denoised_on],
                           ["Noisy input", "Denoised — fault-preservation OFF", "Denoised — fault-preservation ON"]):
    ax.imshow(img, cmap="RdBu_r", vmin=-3, vmax=3, aspect="auto")
    ax.set_title(title)
    ax.set_xlabel("Trace"); ax.set_ylabel("Sample")
plt.tight_layout()
plt.show()

## 3. The reveal: toggle the fault-preservation loss

*"Ordinary denoisers erase the geology. Ours protects it."* Compare the two denoised
panels above around the fault trace(s) — OFF should look visibly smoother/blurrier
right at the fault; ON keeps more of the sharp discontinuity.

In [ ]:
print("Denoising metrics (synthetic, known-clean reference):")
for name, img in [("Noisy (baseline)", noisy), ("Denoised OFF", denoised_off), ("Denoised ON", denoised_on)]:
    print(f"  {name:22s}  PSNR={metrics_mod.psnr(img, clean):6.2f} dB   "
          f"SNR={metrics_mod.snr_db(img, clean):6.2f} dB   SSIM={metrics_mod.ssim(img, clean):.4f}")

## 4. Fault segmentation: noisy vs. denoised input

*"Denoising isn't the goal — finding the trap is, and we find more of them."*

The FaultSeg head below was trained on **clean** synthetic data (mirroring how the real
FaultSeg3D is trained on synthetic volumes before being applied to noisy field data) —
so the gap between its noisy-input and denoised-input performance is a direct measure
of the domain gap that denoising closes.

In [ ]:
prob_noisy = run_faultseg_inference(faultseg, noisy, cfg, device)
prob_denoised = run_faultseg_inference(faultseg, denoised_on, cfg, device)

threshold = cfg["faultseg"]["dice_threshold"]
m_noisy = metrics_mod.fault_metrics(prob_noisy, fault_mask, threshold=threshold)
m_denoised = metrics_mod.fault_metrics(prob_denoised, fault_mask, threshold=threshold)

print(f"{'metric':22s} {'noisy input':>14s} {'denoised input':>16s}")
for field in ["dice", "precision", "recall", "roc_auc", "mean_distance_error"]:
    print(f"{field:22s} {getattr(m_noisy, field):14.3f} {getattr(m_denoised, field):16.3f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
axes[0].imshow(noisy, cmap="RdBu_r", vmin=-3, vmax=3, aspect="auto")
axes[0].set_title("Noisy input")

for ax, section, prob, title in zip(axes[1:], [noisy, denoised_on], [prob_noisy, prob_denoised],
                                     ["Fault picks on NOISY input", "Fault picks on DENOISED input"]):
    ax.imshow(section, cmap="RdBu_r", vmin=-3, vmax=3, aspect="auto")
    overlay = np.where(prob >= threshold, prob, np.nan)
    ax.imshow(overlay, cmap="autumn", alpha=0.6, aspect="auto")
    ax.set_title(title)
plt.tight_layout()
plt.show()

## 5. Diagnostics: F-K spectrum + local-similarity (signal-leakage) map

In [ ]:
fk_noisy = fk_spectrum(torch.from_numpy(noisy)).numpy()
fk_denoised = fk_spectrum(torch.from_numpy(denoised_on)).numpy()

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
axes[0].imshow(fk_noisy, cmap="viridis", aspect="auto"); axes[0].set_title("F-K spectrum — noisy")
axes[1].imshow(fk_denoised, cmap="viridis", aspect="auto"); axes[1].set_title("F-K spectrum — denoised")
plt.tight_layout()
plt.show()

sim_map = metrics_mod.local_similarity_map_fast(noisy, denoised_on, window=9)
plt.figure(figsize=(8, 5))
plt.imshow(sim_map, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
plt.title("Local similarity: (noisy - denoised) vs. denoised\n(near-zero everywhere = no signal leaked into removed noise)")
plt.colorbar(label="correlation")
plt.show()

## 6. Horizon tracking on the denoised section (stretch)

In [ ]:
horizons = track_horizons(denoised_on, n_horizons=cfg["horizon"]["n_horizons"],
                           fault_mask=prob_denoised >= threshold)

plt.figure(figsize=(10, 6))
plt.imshow(denoised_on, cmap="RdBu_r", vmin=-3, vmax=3, aspect="auto")
for h in horizons:
    plt.plot(h, linewidth=2)
plt.title("Auto-tracked horizons on the denoised section")
plt.xlabel("Trace"); plt.ylabel("Sample")
plt.show()

---
### Close

*"Ingests standard SEG-Y, needs no clean labels, drops into ONGC/Oil India's existing
workflow. Fewer dry wells, faster interpretation."*